In [1]:
import torch

In [2]:
# 1. Data gathering
# 2. Data preprocessing
# 3. Feature engineering
# 4. Model training
# 5. Testing

In [3]:
## Dataset used

# https://www.kaggle.com/datasets/mirichoi0218/insurance

In [4]:
!pip install kaggle

In [6]:
import kagglehub

# Download latest version

path = kagglehub.dataset_download("mirichoi0218/insurance")
print(f"Path of dataset files: {path}")

Using Colab cache for faster access to the 'insurance' dataset.
Path of dataset files: /kaggle/input/insurance


In [7]:
import os
print(os.getcwd())

/content


In [8]:
import pandas as pd


In [10]:
os.listdir('/kaggle/input/insurance')

['insurance.csv']

In [11]:
df = pd.read_csv(os.path.join(path, 'insurance.csv'))

In [12]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [15]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [16]:
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [17]:
# Split dataset before encoding

train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 42)

In [18]:
# Encode categorical variable

label_encoder = {}
for col in ['sex', 'smoker', 'region']:
  le = LabelEncoder()
  train_df[col] = le.fit_transform(train_df[col])
  test_df[col] = le.transform(test_df[col])
  label_encoder[col] = le


In [20]:
# Features and target

X_train = train_df.drop(columns = ["charges"])
Y_train = train_df["charges"]
X_test = test_df.drop(columns = ["charges"])
Y_test = test_df["charges"]

In [21]:
print(X_train.head())

      age  sex    bmi  children  smoker  region
560    46    0  19.95         2       0       1
1285   47    0  24.32         0       0       0
1142   52    0  24.86         0       0       2
969    39    0  34.32         5       0       2
486    54    0  21.47         3       0       1


In [22]:
print(X_test.head())

      age  sex     bmi  children  smoker  region
764    45    0  25.175         2       0       0
887    36    0  30.020         0       0       1
890    64    0  26.885         0       1       1
1293   46    1  25.745         3       0       1
259    19    1  31.920         0       1       1


In [26]:
# Normalize features

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [25]:
# Print normalized training features array
print("Normalized X_train:")
print(X_train[:5])
print(f"\nShape: {X_train.shape}")

      age  sex     bmi  children  smoker  region
560    46    0  19.950         2       0       1
1285   47    0  24.320         0       0       0
1142   52    0  24.860         0       0       2
969    39    0  34.320         5       0       2
486    54    0  21.470         3       0       1
...   ...  ...     ...       ...     ...     ...
1095   18    0  31.350         4       0       0
1130   39    0  23.870         5       0       2
1294   58    1  25.175         0       0       0
860    37    0  47.600         2       1       3
1126   55    1  29.900         0       0       3

[1070 rows x 6 columns]


In [27]:
# Convert to tensors

X_train_tensor = torch.tensor(X_train, dtype = torch.float32)
Y_train_tensor = torch.tensor(Y_train.values, dtype = torch.float32).view(-1,1)
X_test_tensor = torch.tensor(X_test, dtype = torch.float32)
Y_test_tensor = torch.tensor(Y_test.values, dtype = torch.float32).view(-1,1)


In [29]:
print(Y_train_tensor)
print(Y_train_tensor.shape)

tensor([[ 9193.8389],
        [ 8534.6719],
        [27117.9941],
        ...,
        [11931.1250],
        [46113.5117],
        [10214.6357]])
torch.Size([1070, 1])


In [45]:
# Define Neural Network

class SimpleNNRegressionModel(nn.Module):
  def __init__(self, input_dim):
    super(SimpleNNRegressionModel, self).__init__()
    self.network = nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Linear(64,128),
        nn.ReLU(),
        nn.Linear(128, 1)
    )

  def forward(self,x):
    return self.network(x)

In [46]:
X_train_tensor.shape

torch.Size([1070, 6])

In [47]:
input_dim = X_train_tensor.shape[1]
model = SimpleNNRegressionModel(input_dim)

In [48]:
print(model)

SimpleNNRegressionModel(
  (network): Sequential(
    (0): Linear(in_features=6, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [49]:
# Loss and Optimizer

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)


In [53]:
# Training Loop
epochs = 10000

for epoch in range(epochs):
  model.train()
  optimizer.zero_grad()
  predictions = model(X_train_tensor)
  loss = criterion(predictions, Y_train_tensor)
  loss.backward()

  optimizer.step()

  if (epoch + 1) % 100 == 0:
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [100/10000], Loss: 2868182.0000
Epoch [200/10000], Loss: 2867226.2500
Epoch [300/10000], Loss: 2848968.5000
Epoch [400/10000], Loss: 2848969.0000
Epoch [500/10000], Loss: 2856546.0000
Epoch [600/10000], Loss: 2849499.7500
Epoch [700/10000], Loss: 2830371.0000
Epoch [800/10000], Loss: 2836550.2500
Epoch [900/10000], Loss: 2776293.7500
Epoch [1000/10000], Loss: 2788190.7500
Epoch [1100/10000], Loss: 2762938.2500
Epoch [1200/10000], Loss: 2769558.2500
Epoch [1300/10000], Loss: 2750627.0000
Epoch [1400/10000], Loss: 2741864.0000
Epoch [1500/10000], Loss: 2757970.5000
Epoch [1600/10000], Loss: 2711230.7500
Epoch [1700/10000], Loss: 2740494.0000
Epoch [1800/10000], Loss: 2711388.7500
Epoch [1900/10000], Loss: 2691474.0000
Epoch [2000/10000], Loss: 2747138.7500
Epoch [2100/10000], Loss: 2671211.5000
Epoch [2200/10000], Loss: 2703815.7500
Epoch [2300/10000], Loss: 2666218.0000
Epoch [2400/10000], Loss: 2642366.2500
Epoch [2500/10000], Loss: 2676184.7500
Epoch [2600/10000], Loss: 2630645.

In [54]:
# Model Evaluation

model.eval()
Y_pred = model(X_test_tensor).detach().numpy()


In [55]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error



Y_test_numpy = Y_test_tensor.numpy()


# Calculate metrics


mse = mean_squared_error(Y_test_numpy, Y_pred)
r2 = r2_score(Y_test_numpy, Y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(Y_test_numpy, Y_pred)

# Print the metrics
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"R-squared (R2) Score: {r2}")
print(f"Mean Absolute Error (MAE): {mae}")

Mean Squared Error (MSE): 51946516.0
Root Mean Squared Error (RMSE): 7207.393148705016
R-squared (R2) Score: 0.665398120880127
Mean Absolute Error (MAE): 4388.328125


In [60]:
def predict_charges(age, sex, bmi, children, smoker, region):
  # Assign the created DataFrame to input_data
  input_data = pd.DataFrame([[age, sex, bmi, children, smoker, region]],
                            columns = ["age", "sex", "bmi", "children", "smoker", "region"])

  # First encode all categorical columns
  for col in ["sex", "smoker", "region"]:
    input_data[col] = label_encoder[col].transform(input_data[col])

  # Scale the complete input data, convert to tensor, and predict outside the loop
  input_data_scaled = scaler.transform(input_data)
  input_data_tensor = torch.tensor(input_data_scaled, dtype = torch.float32)

  model.eval()
  with torch.no_grad():
    predicted_charge = model(input_data_tensor).item()
  return predicted_charge

In [61]:
predicted = predict_charges(19, 'female', 27.9, 0, 'yes', 'southwest')

In [62]:
# Print the predicted charge
print(f"Predicted insurance charge: ${predicted:,.2f}")

Predicted insurance charge: $16,775.86
